# BuildCompiler SynBioHub Authenticated Tutorial

This tutorial shows how to authenticate to SynBioHub with email/password, load inventory collections into BuildCompiler's internal SBOL document, and prepare an assembly workflow from an abstract design URI.

In [1]:
from getpass import getpass
import sbol2

from buildcompiler.api import BuildCompiler

In [2]:
repository_url = 'https://synbiohub.org'
email = input('SynBioHub user/email: ').strip()
password = getpass('SynBioHub password: ')

abstract_design_uri = 'https://synbiohub.org/user/Gon/abstract_design/standard_GFP/1'
collections = [
    'https://synbiohub.org/user/Gon/impl_test/impl_test_collection/1',
    'https://synbiohub.org/user/Gon/Enzyme_Implementations/Enzyme_Implementations_collection/1',
]


In [3]:
doc = sbol2.Document()
compiler = BuildCompiler.from_synbiohub(
    collections=collections,
    repository_url=repository_url,
    email=email,
    password=password,
    sbol_doc=doc,
)

print(f'Loaded objects: {len(doc.componentDefinitions)} component definitions, {len(doc.implementations)} implementations')
print('Repository client:', type(compiler.repository_client).__name__)
print('In-memory auth token available:', compiler.repository_client.auth_token is not None)


Indexing collection: https://synbiohub.org/user/Gon/impl_test/impl_test_collection/1
Indexing collection: https://synbiohub.org/user/Gon/Enzyme_Implementations/Enzyme_Implementations_collection/1
Loaded objects: 55 component definitions, 31 implementations
Repository client: PartShopRepositoryClient
In-memory auth token available: True


## What BuildCompiler does with this setup

- Pulls each collection URI using authenticated `sbol2.PartShop` into `sbol_doc`.
- Reuses the same authenticated pull client for identity-based resolver misses in planning/execution.
- Uses inventory indexing to find compatible plasmids containing abstract-design parts and compatible backbones/reagents.

## Assembly simulation behavior

Legacy Golden Gate simulation now enforces strict digest validation:

- Restriction digest must yield exactly **2 fragments**.
- For part plasmids, the smaller fragment is selected as the insert.
- For the backbone plasmid, the larger fragment is selected as the backbone.
- If digest count is unexpected, simulation fails with a clear error message naming the reactant.
- Successful assembly encodes reagent usage and links generated product implementations with `wasGeneratedBy` to one assembly activity per assembled design product.

In [4]:
# Pull the abstract design into the same in-memory document
if doc.find(abstract_design_uri) is None:
    compiler.repository_client.pull_identity(abstract_design_uri)

abstract_design = doc.find(abstract_design_uri)
print('Abstract design found:', abstract_design is not None, type(abstract_design).__name__)


Abstract design found: True ComponentDefinition


In [5]:
# Run full build using the abstract design object
# Note: this assumes your pulled inventory contains compatible lvl1 part plasmids, backbone, and reagents.
result = compiler.full_build([abstract_design])
print('Build status:', result.status)
print('Stage results:', [(s.stage.value, s.status.value) for s in result.stage_results])


Build status: BuildStatus.SUCCESS
Stage results: [('assembly_lvl1', 'success')]


In [6]:
# Check whether lvl1 assembly was selected
lvl1 = [s for s in result.stage_results if s.stage.value == 'assembly_lvl1']
print('Lvl1 stage entries:', len(lvl1))
if lvl1:
    print('Lvl1 status:', lvl1[0].status.value)


Lvl1 stage entries: 1
Lvl1 status: success


In [8]:
# Access the best-practice SBOL build document
build_doc = result.build_document
print('Build document objects:', len(build_doc.componentDefinitions), 'CDs,', len(build_doc.implementations), 'implementations,', len(build_doc.activities), 'activities')
# build_doc.write('build_result.xml')
build_doc.validate()


Build document objects: 77 CDs, 32 implementations, 2 activities


'Invalid. Exception in thread "main" java.lang.ClassCastException: class java.lang.String cannot be cast to class java.net.URI (java.lang.String and java.net.URI are in module java.base of loader \'bootstrap\')\n\tat org.sbolstandard.core2.SBOLReader.parseComponentDefinition(SBOLReader.java:1921)\n\tat org.sbolstandard.core2.SBOLReader.readTopLevelDocs(SBOLReader.java:1078)\n\tat org.sbolstandard.core2.SBOLReader.read(SBOLReader.java:745)\n\tat org.sbolstandard.core2.SBOLReader.read(SBOLReader.java:632)\n\tat org.sbolstandard.core2.SBOLReader.read(SBOLReader.java:519)\n\tat org.sbolstandard.core2.SBOLReader.read(SBOLReader.java:444)\n\tat org.sbolstandard.core2.SBOLReader.read(SBOLReader.java:429)\n\tat org.sbolstandard.core2.SBOLValidate.validate(SBOLValidate.java:2772)\n\tat org.sbolstandard.core2.SBOLValidate.main(SBOLValidate.java:3100)\n'

In [ ]:
# Enable detailed reporting and inspect full diagnostic report when builds fail
from buildcompiler.api import BuildOptions

debug_options = BuildOptions()
debug_options.reporting.include_detailed_report = True
debug_result = compiler.full_build([abstract_design], options=debug_options)
print('Debug status:', debug_result.status)
print('Missing inputs:', [m.missing_identity for m in debug_result.missing_inputs])
print('Warnings:', [w.message if hasattr(w, 'message') else str(w) for w in debug_result.warnings])
debug_result.report


In [ ]:
# Validate full build document
build_doc.validate()
print('Full build document validation completed.')


In [ ]:
# Build a focused assembly-only document and validate
assembly_doc = sbol2.Document()
for activity in build_doc.activities:
    if 'assembly' in activity.displayId.lower():
        activity.copy(assembly_doc)

for impl in build_doc.implementations:
    if getattr(impl, 'wasGeneratedBy', None):
        impl.copy(assembly_doc)
        built_obj = build_doc.find(impl.built)
        if built_obj is not None:
            built_obj.copy(assembly_doc)

assembly_doc.validate()
print('Assembly-only document validation completed.')
